[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/06_multihead_attention_solution.ipynb)

# ✅ Solution: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Shape suffixes (Noam)
`B`=batch, `L`=query length, `M`=KV length, `D`=`d_model`, `H`=heads, `K`=`d_k`

### Signature
```python
class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs): ...
    def __call__(self, q_BLD, k_BMD, v_BMD) -> jax.Array: ...
```

### Requirements
- Use `nnx.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `K = d_model // num_heads` per head
- `q_BLD` is `(B, L, D)`; `k_BMD` / `v_BMD` are `(B, M, D)` (support `L != M`)
- Do **NOT** use a built-in multi-head attention module
- You **may** use `jax.nn.softmax` and `jnp.matmul`

### Steps
1. Project: `q_BLD = self.W_q(q_BLD)`, same for k/v
2. Reshape/transpose to `q_BHLK`, `k_BHMK`, `v_BHMK`
3. Scaled dot-product attention → `attn_BHLK`
4. Concat heads → `concat_BLD`
5. Output projection: `self.W_o(concat_BLD)`


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import math


In [ ]:
# ✅ SOLUTION

import jax, jax.numpy as jnp, math
from flax import nnx
class MultiHeadAttention(nnx.Module):
    """Shape suffixes: B=batch, L=query len, M=KV len, D=d_model, H=heads, K=d_k."""
    def __init__(self, d_model, num_heads, *, rngs):
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_k = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_v = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_o = nnx.Linear(d_model, d_model, rngs=rngs)
    def __call__(self, q_BLD, k_BMD, v_BMD):
        B, L, _ = q_BLD.shape
        M = k_BMD.shape[1]
        H, K = self.num_heads, self.d_k
        q_BHLK = self.W_q(q_BLD).reshape(B, L, H, K).transpose(0, 2, 1, 3)
        k_BHMK = self.W_k(k_BMD).reshape(B, M, H, K).transpose(0, 2, 1, 3)
        v_BHMK = self.W_v(v_BMD).reshape(B, M, H, K).transpose(0, 2, 1, 3)
        scores_BHLM = q_BHLK @ jnp.swapaxes(k_BHMK, -2, -1) / math.sqrt(K)
        attn_BHLK = jax.nn.softmax(scores_BHLM, axis=-1) @ v_BHMK
        concat_BLD = attn_BHLK.transpose(0, 2, 1, 3).reshape(B, L, H * K)
        return self.W_o(concat_BLD)


In [ ]:
# Verify
print(MultiHeadAttention)


In [ ]:
from jax_judge import check
check("mha")
